# 03 — Análise de Receita
**Olist Analytics Hub | Etapa 3 de 7**

**Objetivo:** Entender a evolução da receita ao longo do tempo, as categorias mais lucrativas, o comportamento de pagamento e a sazonalidade das vendas.

---
### Perguntas que este notebook responde
1. Como o GMV evoluiu mês a mês?
2. Quais categorias geram mais receita?
3. Qual o ticket médio por método de pagamento?
4. Existe sazonalidade semanal ou horária nas vendas?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

from src.data_cleaner import load_processed
from src.visualizations import setup_style, save_fig, format_brl, time_series_chart, bar_chart, heatmap_chart
from src.config import COLORS

setup_style()
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✓ Ambiente configurado.')

In [ ]:
df = load_processed('orders_enriched')

# Apenas pedidos entregues para análises de receita
delivered = df[df['is_delivered']].copy()

print(f'Total de pedidos: {len(df):,}')
print(f'Pedidos entregues: {len(delivered):,} ({len(delivered)/len(df):.1%})')
print(f'Período: {delivered["order_purchase_timestamp"].min().date()} → {delivered["order_purchase_timestamp"].max().date()}')
print(f'GMV total: {format_brl(delivered["revenue"].sum())}')

## 1. Evolução Mensal do GMV

In [ ]:
# GMV mensal
gmv_monthly = (
    delivered
    .groupby(delivered['order_purchase_timestamp'].dt.to_period('M'))
    .agg(
        gmv=('revenue', 'sum'),
        pedidos=('order_id', 'count'),
        ticket_medio=('revenue', 'mean'),
    )
    .reset_index()
)
gmv_monthly['mes'] = gmv_monthly['order_purchase_timestamp'].dt.to_timestamp()

# Remove meses incompletos (primeiro e último)
gmv_monthly = gmv_monthly.iloc[1:-1]

print(f'Meses analisados: {len(gmv_monthly)}')
print(f'Melhor mês: {gmv_monthly.loc[gmv_monthly["gmv"].idxmax(), "mes"].strftime("%b/%Y")} — {format_brl(gmv_monthly["gmv"].max())}')
display(gmv_monthly[['mes', 'gmv', 'pedidos', 'ticket_medio']].tail(6))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# GMV mensal
axes[0].fill_between(gmv_monthly['mes'], gmv_monthly['gmv'], alpha=0.2, color=COLORS['primary'])
axes[0].plot(gmv_monthly['mes'], gmv_monthly['gmv'], color=COLORS['primary'], linewidth=2.5, marker='o', markersize=5)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
axes[0].set_title('GMV Mensal — Olist 2017-2018')
axes[0].set_xlabel('')
axes[0].set_ylabel('GMV')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

# Volume de pedidos mensais
axes[1].bar(gmv_monthly['mes'], gmv_monthly['pedidos'], width=20, color=COLORS['secondary'], alpha=0.85)
axes[1].set_title('Volume Mensal de Pedidos')
axes[1].set_xlabel('')
axes[1].set_ylabel('Pedidos')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
save_fig('03_gmv_mensal')
plt.show()

## 2. Top Categorias por Receita

In [ ]:
cat_revenue = (
    delivered
    .dropna(subset=['main_category'])
    .groupby('main_category')
    .agg(
        receita=('revenue', 'sum'),
        pedidos=('order_id', 'count'),
        ticket_medio=('revenue', 'mean'),
        nota_media=('review_score', 'mean'),
    )
    .sort_values('receita', ascending=False)
)
cat_revenue['share_pct'] = cat_revenue['receita'] / cat_revenue['receita'].sum() * 100

print('Top 10 Categorias por Receita:')
display(
    cat_revenue.head(10)
    .style
    .format({'receita': 'R$ {:,.0f}', 'ticket_medio': 'R$ {:,.2f}', 'nota_media': '{:.2f}', 'share_pct': '{:.1f}%'})
    .background_gradient(subset=['receita'], cmap='Blues')
)

In [ ]:
top10 = cat_revenue.head(10).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Receita por categoria
bars = axes[0].barh(top10['main_category'], top10['receita'] / 1e6, color=COLORS['primary'], alpha=0.85)
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 0.02, bar.get_y() + bar.get_height() / 2, f'R$ {w:.1f}M', va='center', fontsize=8)
axes[0].invert_yaxis()
axes[0].set_title('Top 10 Categorias por Receita')
axes[0].set_xlabel('Receita (R$ milhões)')

# Ticket médio por categoria
top10_ticket = cat_revenue.head(10).sort_values('ticket_medio', ascending=True).reset_index()
axes[1].barh(top10_ticket['main_category'], top10_ticket['ticket_medio'], color=COLORS['accent'], alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_title('Ticket Médio — Top 10 Categorias')
axes[1].set_xlabel('Ticket Médio (R$)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

plt.tight_layout()
save_fig('03_top_categorias')
plt.show()

## 3. Análise por Método de Pagamento

In [ ]:
payment_stats = (
    delivered
    .dropna(subset=['payment_type'])
    .groupby('payment_type')
    .agg(
        pedidos=('order_id', 'count'),
        receita=('payment_value', 'sum'),
        ticket_medio=('payment_value', 'mean'),
        parcelas_media=('payment_installments', 'mean'),
    )
    .sort_values('pedidos', ascending=False)
)
payment_stats['share_pedidos'] = payment_stats['pedidos'] / payment_stats['pedidos'].sum() * 100

print('Análise por Método de Pagamento:')
display(
    payment_stats
    .style
    .format({'receita': 'R$ {:,.0f}', 'ticket_medio': 'R$ {:,.2f}',
             'parcelas_media': '{:.1f}', 'share_pedidos': '{:.1f}%'})
    .background_gradient(subset=['pedidos'], cmap='Greens')
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ps = payment_stats.reset_index()
palette = COLORS['palette'][:len(ps)]

# Pizza de participação
axes[0].pie(ps['pedidos'], labels=ps['payment_type'], autopct='%1.1f%%',
            colors=palette, startangle=140)
axes[0].set_title('Participação por Método (pedidos)')

# Ticket médio
axes[1].bar(ps['payment_type'], ps['ticket_medio'], color=palette, alpha=0.85)
for i, (_, row) in enumerate(ps.iterrows()):
    axes[1].text(i, row['ticket_medio'] + 5, format_brl(row['ticket_medio']), ha='center', fontsize=8)
axes[1].set_title('Ticket Médio por Método')
axes[1].set_ylabel('R$')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))

# Parcelas médias (apenas cartão)
credit = ps[ps['payment_type'] == 'credit_card']
if not credit.empty:
    parcel_dist = (
        delivered[delivered['payment_type'] == 'credit_card']
        ['payment_installments']
        .value_counts()
        .sort_index()
        .head(12)
    )
    axes[2].bar(parcel_dist.index.astype(str), parcel_dist.values, color=COLORS['primary'], alpha=0.85)
    axes[2].set_title('Distribuição de Parcelas — Cartão de Crédito')
    axes[2].set_xlabel('Número de Parcelas')
    axes[2].set_ylabel('Pedidos')
else:
    axes[2].set_visible(False)

plt.tight_layout()
save_fig('03_pagamentos')
plt.show()

## 4. Sazonalidade Semanal e Horária

In [ ]:
# Ordem dos dias da semana
dias_ordem = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dias_pt = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']

pedidos_por_dia = (
    delivered
    .groupby('purchase_day_of_week')['order_id']
    .count()
    .reindex(dias_ordem)
)

pedidos_por_hora = (
    delivered
    .groupby('purchase_hour')['order_id']
    .count()
)

receita_por_dia = (
    delivered
    .groupby('purchase_day_of_week')['revenue']
    .sum()
    .reindex(dias_ordem)
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Pedidos por dia da semana
colors_dias = [COLORS['primary'] if i < 5 else COLORS['secondary'] for i in range(7)]
axes[0, 0].bar(dias_pt, pedidos_por_dia.values, color=colors_dias, alpha=0.85)
axes[0, 0].set_title('Volume de Pedidos por Dia da Semana')
axes[0, 0].set_ylabel('Pedidos')

# Receita por dia da semana
axes[0, 1].bar(dias_pt, receita_por_dia.values / 1e6, color=colors_dias, alpha=0.85)
axes[0, 1].set_title('Receita por Dia da Semana')
axes[0, 1].set_ylabel('Receita (R$ milhões)')

# Pedidos por hora
axes[1, 0].fill_between(pedidos_por_hora.index, pedidos_por_hora.values, alpha=0.25, color=COLORS['accent'])
axes[1, 0].plot(pedidos_por_hora.index, pedidos_por_hora.values, color=COLORS['accent'], linewidth=2.5, marker='o', markersize=4)
axes[1, 0].axvspan(22, 24, alpha=0.1, color='gray', label='Horário noturno')
axes[1, 0].axvspan(0, 6, alpha=0.1, color='gray')
axes[1, 0].set_title('Volume de Pedidos por Hora do Dia')
axes[1, 0].set_xlabel('Hora')
axes[1, 0].set_ylabel('Pedidos')
axes[1, 0].set_xticks(range(0, 24, 2))

# Heatmap hora × dia da semana
heatmap_data = (
    delivered
    .groupby(['purchase_day_of_week', 'purchase_hour'])['order_id']
    .count()
    .unstack(fill_value=0)
    .reindex(dias_ordem)
)
heatmap_data.index = dias_pt
sns.heatmap(heatmap_data, ax=axes[1, 1], cmap='YlOrRd', linewidths=0,
            cbar_kws={'shrink': 0.8}, xticklabels=2)
axes[1, 1].set_title('Heatmap Pedidos: Hora × Dia da Semana')
axes[1, 1].set_xlabel('Hora')
axes[1, 1].set_ylabel('')

plt.tight_layout()
save_fig('03_sazonalidade')
plt.show()

## 5. Evolução do Ticket Médio por Trimestre

In [ ]:
quarterly = (
    delivered
    .assign(trimestre=delivered['order_purchase_timestamp'].dt.to_period('Q'))
    .groupby('trimestre')
    .agg(
        gmv=('revenue', 'sum'),
        pedidos=('order_id', 'count'),
        ticket_medio=('revenue', 'mean'),
    )
    .reset_index()
)
quarterly['trimestre_str'] = quarterly['trimestre'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(quarterly['trimestre_str'], quarterly['gmv'] / 1e6, color=COLORS['primary'], alpha=0.85)
axes[0].set_title('GMV por Trimestre')
axes[0].set_ylabel('Receita (R$ milhões)')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

axes[1].plot(quarterly['trimestre_str'], quarterly['ticket_medio'],
             color=COLORS['secondary'], linewidth=2.5, marker='s', markersize=7)
for i, row in quarterly.iterrows():
    axes[1].annotate(format_brl(row['ticket_medio']),
                     (row['trimestre_str'], row['ticket_medio']),
                     textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
axes[1].set_title('Ticket Médio por Trimestre')
axes[1].set_ylabel('Ticket Médio (R$)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
save_fig('03_trimestral')
plt.show()

## 6. Sumário e Insights

| Métrica | Valor |
|---------|-------|
| GMV Total (pedidos entregues) | *(calculado acima)* |
| Melhor mês | *(calculado acima)* |
| Top categoria por receita | *(calculado acima)* |
| Método de pagamento dominante | cartão de crédito (~74%) |
| Horário de pico | 10h–16h, dias úteis |

### Próximo Notebook
`04_analise_clientes.ipynb` — distribuição geográfica, padrões de comportamento e taxa de recompra.